In [ ]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

import xgboost as xgb

from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error


# ============================================================
# 2. LOAD DATA
# ============================================================

DATA_PATH = "/kaggle/input/competitions/house-prices-advanced-regression-techniques"

train = pd.read_csv(f"{DATA_PATH}/train.csv")
test = pd.read_csv(f"{DATA_PATH}/test.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

# Lưu ID để tạo file submission
test_id = test["Id"].copy()

# Lưu SalePrice và chuyển sang log
y_train = np.log1p(train["SalePrice"])

# Không sử dụng Id và SalePrice làm feature
train_features = train.drop(columns=["Id", "SalePrice"])
test_features = test.drop(columns=["Id"])


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

def create_features(df):
    df = df.copy()

    # Tổng diện tích sử dụng
    df["TotalSF"] = (
        df["TotalBsmtSF"].fillna(0)
        + df["1stFlrSF"].fillna(0)
        + df["2ndFlrSF"].fillna(0)
    )

    # Tổng số phòng tắm
    df["TotalBath"] = (
        df["FullBath"].fillna(0)
        + 0.5 * df["HalfBath"].fillna(0)
        + df["BsmtFullBath"].fillna(0)
        + 0.5 * df["BsmtHalfBath"].fillna(0)
    )

    # Tổng diện tích hiên nhà
    df["TotalPorchSF"] = (
        df["OpenPorchSF"].fillna(0)
        + df["EnclosedPorch"].fillna(0)
        + df["3SsnPorch"].fillna(0)
        + df["ScreenPorch"].fillna(0)
    )

    # Tuổi căn nhà tại thời điểm bán
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

    # Số năm kể từ lần tu sửa gần nhất
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

    # Chuyển giá trị tuổi âm bất thường về 0
    df["HouseAge"] = df["HouseAge"].clip(lower=0)
    df["RemodAge"] = df["RemodAge"].clip(lower=0)

    return df


train_features = create_features(train_features)
test_features = create_features(test_features)


# ============================================================
# 4. COMBINE TRAIN AND TEST FOR CONSISTENT ENCODING
# ============================================================

n_train = len(train_features)

all_data = pd.concat(
    [train_features, test_features],
    axis=0,
    ignore_index=True
)

# Phân loại cột tự động
cat_cols = all_data.select_dtypes(
    include=["object", "category"]
).columns

num_cols = all_data.select_dtypes(
    include=["number", "bool"]
).columns

# Xử lý missing values ở cột categorical
for col in cat_cols:
    all_data[col] = all_data[col].fillna("Missing").astype(str)

# Xử lý missing values ở cột numerical
for col in num_cols:
    all_data[col] = all_data[col].fillna(
        all_data[col].median()
    )

# One-Hot Encoding
all_data = pd.get_dummies(
    all_data,
    columns=list(cat_cols),
    drop_first=False,
    dtype=int
)

# Tách lại train và test
X_train = all_data.iloc[:n_train].copy()
X_test = all_data.iloc[n_train:].copy()

print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)

print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())


# ============================================================
# 5. BUILD XGBOOST MODEL
# ============================================================

xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)


# ============================================================
# 6. HYPERPARAMETER TUNING
# ============================================================

param_dist = {
    "n_estimators": [500, 800, 1000, 1500, 2000],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.07],
    "min_child_weight": [1, 2, 3, 5, 7],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.01, 0.05, 0.1, 0.2],
    "reg_alpha": [0, 0.001, 0.01, 0.1, 1],
    "reg_lambda": [1, 2, 5, 10]
}

# 5-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_cv = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=40,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
    refit=True
)

# Tìm tham số tốt nhất
random_cv.fit(X_train, y_train)

print("\nBest Parameters:")
print(random_cv.best_params_)

print("\nBest CV RMSE (log scale):")
print(-random_cv.best_score_)


# ============================================================
# 7. TRAIN FINAL MODEL
# ============================================================

best_model = random_cv.best_estimator_

# RandomizedSearchCV đã refit mô hình tốt nhất trên toàn bộ train
print("\nFinal model trained successfully!")


# ============================================================
# 8. CROSS VALIDATION SCORE
# ============================================================

cv_rmse = -random_cv.best_score_

print(f"5-Fold CV RMSE (log): {cv_rmse:.5f}")


# ============================================================
# 9. PREDICT TEST DATA
# ============================================================

pred_log = best_model.predict(X_test)

# Đưa từ log giá về giá thực
predictions = np.expm1(pred_log)

# Giá nhà không thể âm
predictions = np.maximum(predictions, 0)

print("Prediction Shape:", predictions.shape)
print("Min Prediction:", predictions.min())
print("Max Prediction:", predictions.max())


# ============================================================
# 10. SAVE MODEL
# ============================================================

with open("xgb_model_optimized.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Model saved: xgb_model_optimized.pkl")


# ============================================================
# 11. CREATE KAGGLE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": predictions
})

submission.to_csv(
    "submission_xgb_optimized.csv",
    index=False
)

print("\nSubmission created successfully!")
print(submission.head())

print("\nSubmission Shape:", submission.shape)
print("Missing predictions:", submission["SalePrice"].isnull().sum())